# Model & Quality Monitors Plus Dashboard & Reports

This code adds Data & Model Quality Monitors in the AWS environment to track shifts or degredation in the data or model. The code here follows the following steps: 

1. Environment Setup
2. Endpoint Deployment with Data Capture Enabled
3. Data Monitor Setup, plus Monitoring Schedule
4. Model Monitor Setup, plus Monitoring Schedule
5. Executing Models to see what production data will be like
6. Infrastructure Monitors and Alarms
7. Dashboard
8. Verification code, to ensure monitors are workings as expected
9. Monitor Report Download
10. Cleanup

In a production environment, where data is not static, it is critical to monitor different systems in order to ensure proper functionality. By monitoring the infrastructure, as well as the data, and the model itself, we are able to track performance and alert the appropriate team if anything isn't operating as intended. 

Attribution: This code was made with the help of AWS tutorials, reference of lab resources in AAI 540, as well as both Claude Code and Perplexity accessed February, 2026.

## 1. Environment Setup

In [1]:
#1.1 Library Imports
import sagemaker
from sagemaker import Session
from sagemaker.model import Model
from sagemaker.predictor import Predictor
from sagemaker import image_uris, get_execution_role
from sagemaker.model_monitor import (
    DefaultModelMonitor,
    ModelQualityMonitor,
    DatasetFormat,
    CronExpressionGenerator,
    EndpointInput,
)
from sagemaker.s3 import S3Uploader
from sagemaker.s3 import S3Downloader

import os
import s3fs
import boto3
import pandas as pd
import numpy as np
import json
import time
from datetime import datetime, timedelta, timezone
from io import StringIO
from pathlib import Path
from threading import Thread
from time import sleep
from urllib.parse import quote

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [2]:
#1.2 Configuration

region = "us-east-1"
sm_client = boto3.client("sagemaker", region_name=region)
cw_client = boto3.client("cloudwatch", region_name=region)
s3_client = boto3.client("s3", region_name=region)
role = sagemaker.get_execution_role()
session = sagemaker.Session(boto_session=boto3.Session(region_name=region)) 
bucket = session.default_bucket()  # Dynamically fetches/creates
prefix = "models/benchmarks"

In [3]:
#Confirm access to bucket

# Create the test directory and file
os.makedirs("test_data", exist_ok=True)
with open("test_data/upload-test-file.txt", "w") as f:
    f.write("This is a test file to verify S3 access.")

#Upload file
s3_uri = S3Uploader.upload("test_data/upload-test-file.txt", f"s3://{bucket}/test_upload")
print(f"Success! File uploaded to: {s3_uri}")
print("You are all set to proceed.")

Success! File uploaded to: s3://sagemaker-us-east-1-513691803389/test_upload/upload-test-file.txt
You are all set to proceed.


## 2. Deploy model with Data Capture to Endpoint

In [4]:
#2.1 Locate model artifacts
local_base = Path("/tmp/Models/benchmarks")
s3_base = f"s3://{bucket}/models/benchmarks"

xgb_paths = {
    "local_tar.gz": local_base / "xgboost/model.tar.gz",
    "s3_tar.gz": f"{s3_base}/xgboost/model.tar.gz",
}

# For loading/testing locally
local_model_path = xgb_paths["local_tar.gz"]
if local_model_path.exists():
    print("Local model exists for testing")
else:
    print("No local; using S3")

# ALWAYS use S3 for model_data in Model/deploy
model_data_uri = xgb_paths["s3_tar.gz"]

# Auto-upload local to S3 if needed
if local_model_path.exists():
    s3_client = boto3.client('s3', region_name=region)
    model_key = model_data_uri.split('/', 3)[-1]  # Extract key
    s3_client.upload_file(str(local_model_path), bucket, model_key)
    print(f"Uploaded local to {model_data_uri}")

No local; using S3


In [5]:
#2.2 Container, pre-built Docker image for AWS that runs XGBoost for training, batch transform, or inference
xgboost_container = image_uris.retrieve(
    framework="xgboost",
    region=region,
    version="1.7-1",
)

print(f"XGBoost image: {xgboost_container}")

XGBoost image: 683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-xgboost:1.7-1


In [6]:
#2.3 Model & Endpoint
xg_model = sagemaker.Model(
    image_uri=xgboost_container,  
    model_data=model_data_uri,    
    role=role,
    sagemaker_session=session
)

xgb_endpoint_name = f"xgb-benchmark-endpoint-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

data_capture_prefix = f"{prefix}/datacapture"
data_capture_s3_uri = f"s3://{bucket}/{data_capture_prefix}"

# Deploy with data capture enabled
xg_predictor = xg_model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    endpoint_name=xgb_endpoint_name,
    data_capture_config=sagemaker.model_monitor.DataCaptureConfig(
        enable_capture=True,
        sampling_percentage=100,
        destination_s3_uri=data_capture_s3_uri,
        capture_options=["REQUEST", "RESPONSE"],
    ),
)

print("Endpoint:", xgb_endpoint_name)

------!Endpoint: xgb-benchmark-endpoint-20260215-045008


In [7]:
#2.4 Wait until endpoint is InService
print("Waiting for endpoint to be ready...")
while True:
    resp = sm_client.describe_endpoint(EndpointName=xgb_endpoint_name)
    status = resp["EndpointStatus"]
    print(" Status:", status)
    if status == "InService":
        print("Endpoint is ready!")
        break
    if status == "Failed":
        raise RuntimeError(f"Endpoint deployment failed: {resp.get('FailureReason')}")
    time.sleep(30)

Waiting for endpoint to be ready...
 Status: InService
Endpoint is ready!


## 3. DQ Data Monitor
Data Quality

In [8]:
#3.0 Dynamic URI Finder
bucket = sagemaker.Session().default_bucket()  # Confirms your bucket
prefix = "models/benchmarks"
baseline_key = f"{prefix}/baseline_normalized.csv"
baseline_dataset_uri = f"s3://{bucket}/{baseline_key}"

# Verify exists
s3_client = boto3.client('s3')
s3_client.head_object(Bucket=bucket, Key=baseline_key)
print(f"✅ Dataset: {baseline_dataset_uri}")

✅ Dataset: s3://sagemaker-us-east-1-513691803389/models/benchmarks/baseline_normalized.csv


In [9]:
#3.1: Create Data Quality Monitor
dq_monitor = DefaultModelMonitor(
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    volume_size_in_gb=20,
    max_runtime_in_seconds=3600,
    sagemaker_session=session,
)

In [10]:
#3.2: Generate baseline using your data (creates stats/constraints)
dq_baseline_uri = f"s3://{bucket}/{prefix}/monitoring/dq-baseline"

# Run baselining job
dq_baseline_job = dq_monitor.suggest_baseline(
    baseline_dataset_uri,
    dataset_format=DatasetFormat.csv(header=True),
    output_s3_uri=dq_baseline_uri,
    wait=True,
    logs=False,
)

print("✅ DQ baseline created!")

INFO:sagemaker:Creating processing-job with name baseline-suggestion-job-2026-02-15-04-53-41-421


...........................................................!✅ DQ baseline created!


In [11]:
#3.3: Get the generated stats/constraints URIs
try:
    dq_stats_uri = dq_monitor.latest_baselining_job.baseline_statistics.file_name
    dq_constraints_uri = dq_monitor.latest_baselining_job.suggested_constraints.file_name
except:
    # Fallback: find files manually
    s3 = boto3.client("s3")
    resp = s3.list_objects_v2(Bucket=bucket, Prefix=f"{prefix}/monitoring/dq-baseline/")
    for obj in resp.get("Contents", []):
        if "statistics.json" in obj["Key"]:
            dq_stats_uri = f"s3://{bucket}/{obj['Key']}"
        if "constraints.json" in obj["Key"]:
            dq_constraints_uri = f"s3://{bucket}/{obj['Key']}"

print("Data Quality Stats:", dq_stats_uri)
print("Data Quality Constraints:", dq_constraints_uri)

Data Quality Stats: s3://sagemaker-us-east-1-513691803389/models/benchmarks/monitoring/dq-baseline/statistics.json
Data Quality Constraints: s3://sagemaker-us-east-1-513691803389/models/benchmarks/monitoring/dq-baseline/constraints.json


In [12]:
# 3.4: Create monitoring schedule using new baseline
schedule_name_xgb_dq = "xgb-data-quality-schedule"

try:
    dq_monitor.delete_monitoring_schedule(schedule_name_xgb_dq)
except:
    pass  # No existing schedule

dq_monitor.create_monitoring_schedule(
    monitor_schedule_name=schedule_name_xgb_dq,
    endpoint_input=xgb_endpoint_name,
    output_s3_uri=f"s3://{bucket}/{prefix}/monitoring/data-quality/xgb",
    statistics=dq_stats_uri,
    constraints=dq_constraints_uri,
    schedule_cron_expression=CronExpressionGenerator.hourly(),
    enable_cloudwatch_metrics=True,
)

print("✅ Data quality schedule live:", schedule_name_xgb_dq)

INFO:sagemaker.model_monitor.model_monitoring:Creating Monitoring Schedule with name: xgb-data-quality-schedule


✅ Data quality schedule live: xgb-data-quality-schedule


In [13]:
dq_monitor.describe_schedule()

{'MonitoringScheduleArn': 'arn:aws:sagemaker:us-east-1:513691803389:monitoring-schedule/xgb-data-quality-schedule',
 'MonitoringScheduleName': 'xgb-data-quality-schedule',
 'MonitoringScheduleStatus': 'Pending',
 'MonitoringType': 'DataQuality',
 'CreationTime': datetime.datetime(2026, 2, 15, 4, 58, 44, 990000, tzinfo=tzlocal()),
 'LastModifiedTime': datetime.datetime(2026, 2, 15, 4, 58, 45, 57000, tzinfo=tzlocal()),
 'MonitoringScheduleConfig': {'ScheduleConfig': {'ScheduleExpression': 'cron(0 * ? * * *)'},
  'MonitoringJobDefinitionName': 'data-quality-job-definition-2026-02-15-04-58-43-978',
  'MonitoringType': 'DataQuality'},
 'EndpointName': 'xgb-benchmark-endpoint-20260215-045008',
 'LastMonitoringExecutionSummary': {'MonitoringScheduleName': 'xgb-data-quality-schedule',
  'ScheduledTime': datetime.datetime(2026, 2, 15, 4, 0, tzinfo=tzlocal()),
  'CreationTime': datetime.datetime(2026, 2, 15, 4, 1, 22, 212000, tzinfo=tzlocal()),
  'LastModifiedTime': datetime.datetime(2026, 2, 15

In [14]:
dq_executions = dq_monitor.list_executions()
dq_executions

## 4. MQ Model Monitor 
Model Quality

In [15]:
#4.1 Create Model Quality Monitor
mq_monitor = ModelQualityMonitor(
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    volume_size_in_gb=20,
    max_runtime_in_seconds=1800,
    sagemaker_session=session,
)

INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.


In [16]:
#4.2 Create baseline dataset and traffic simulation files

print("Loading baseline data...")
baseline_df = pd.read_csv(f"s3://{bucket}/{prefix}/baseline_normalized.csv")

# Sample the SAME 300 samples (same random_state for consistency)
BASELINE_SAMPLE_SIZE = 300
baseline_sample = baseline_df.sample(n=min(BASELINE_SAMPLE_SIZE, len(baseline_df)), 
                                     random_state=42)

print(f"Using {BASELINE_SAMPLE_SIZE} samples for baseline")

# Prepare features for prediction
X_baseline = baseline_sample.drop("target", axis=1)

# Get predictions from endpoint
print(f"Making predictions for {len(X_baseline)} samples...")
csv_payload = X_baseline.to_csv(header=False, index=False)

predictor = Predictor(endpoint_name=xgb_endpoint_name, sagemaker_session=session)
response = predictor.predict(
    data=csv_payload,
    initial_args={"ContentType": "text/csv", "Accept": "text/csv"},
)

# Parse multi-class predictions
predictions_text = response.decode("utf-8").strip().split("\n")
prob_matrix = np.array([[float(x) for x in line.split(",") if x] for line in predictions_text])

# Get predicted labels and probabilities
pred_labels = np.argmax(prob_matrix, axis=1)
pred_probabilities = np.max(prob_matrix, axis=1)

print(f"✓ Predictions complete")
print(f"  Prediction distribution: {np.bincount(pred_labels)}")

# Create baseline dataset for Model Quality monitoring
mq_baseline_df = pd.DataFrame({
    'prediction': pred_labels,
    'probability': pred_probabilities,
    'ground_truth_label': baseline_sample['target'].values
})

# Add all class probabilities for detailed monitoring
for i in range(prob_matrix.shape[1]):
    mq_baseline_df[f'probability_class_{i}'] = prob_matrix[:, i]

print(f"\n📊 Baseline dataset preview:")
print(mq_baseline_df.head(10))
print(f"\nBaseline statistics:")
print(mq_baseline_df.describe())

# Upload baseline to S3 for Model Quality monitoring
mq_baseline_key = f"{prefix}/mq_baseline.csv"
csv_buffer = StringIO()
mq_baseline_df.to_csv(csv_buffer, index=False)

s3_client.put_object(
    Bucket=bucket,
    Key=mq_baseline_key,
    Body=csv_buffer.getvalue(),
    ContentType="text/csv",
)

mq_baseline_uri = f"s3://{bucket}/{mq_baseline_key}"
print(f"\n✅ Model Quality baseline uploaded:")
print(f"   URI: {mq_baseline_uri}")
print(f"   Samples: {len(mq_baseline_df)}")

# Save files locally for traffic simulation
print(f"\n📁 Creating local files for traffic simulation...")
os.makedirs("test_data", exist_ok=True)

# Save features only (for endpoint inference)
X_baseline.to_csv("test_data/traffic_data.csv", index=False)

# Save with labels (for ground truth reference)
baseline_sample.to_csv("test_data/traffic_with_labels.csv", index=False)

print(f"✅ Traffic simulation files created:")
print(f"   - test_data/traffic_data.csv (features only)")
print(f"   - test_data/traffic_with_labels.csv (with labels)")
print(f"   - {len(baseline_sample)} samples")

print("\n" + "="*80)
print("🎉 BASELINE SETUP COMPLETE")
print("="*80)
print(f"✅ Model Quality baseline created and uploaded")
print(f"✅ Traffic simulation files ready")
print(f"✅ Using consistent {BASELINE_SAMPLE_SIZE} samples (random_state=42)")
print("="*80)

Loading baseline data...
Using 300 samples for baseline
Making predictions for 300 samples...
✓ Predictions complete
  Prediction distribution: [  0   0  76   0   2 222]

📊 Baseline dataset preview:
   prediction  probability  ground_truth_label  probability_class_0  \
0           5     0.839737                   5             0.030183   
1           5     0.882021                   4             0.008201   
2           5     0.554950                   1             0.075996   
3           5     0.646829                   5             0.019597   
4           2     0.730500                   2             0.011923   
5           5     0.688689                   0             0.025658   
6           5     0.556096                   0             0.018873   
7           5     0.617959                   4             0.031238   
8           5     0.556317                   4             0.048114   
9           2     0.668872                   4             0.013717   

   probability_clas

In [17]:
#4.3 Inspect predictions
# Read from S3 and display
mq_baseline_uri = f"s3://{bucket}/{prefix}/mq_baseline.csv"
df = pd.read_csv(mq_baseline_uri)

print("First 10 predictions:")
print(df.head(10))

print("\nDataset info:")
print(f"Total samples: {len(df)}")
print(f"\nColumn names: {df.columns.tolist()}")
print(f"\nPrediction distribution:")
print(df['prediction'].value_counts())

print("\nBasic statistics:")
print(df.describe())

First 10 predictions:
   prediction  probability  ground_truth_label  probability_class_0  \
0           5     0.839737                   5             0.030183   
1           5     0.882021                   4             0.008201   
2           5     0.554950                   1             0.075996   
3           5     0.646829                   5             0.019597   
4           2     0.730500                   2             0.011923   
5           5     0.688689                   0             0.025658   
6           5     0.556096                   0             0.018873   
7           5     0.617959                   4             0.031238   
8           5     0.556317                   4             0.048114   
9           2     0.668872                   4             0.013717   

   probability_class_1  probability_class_2  probability_class_3  \
0             0.008047             0.023991             0.008226   
1             0.003313             0.028684             0.00

In [18]:
#4.4 Run baselining from endpoint deployed model predictions locally
mq_baseline_folder = f"s3://{bucket}/{prefix}/mq-baseline"

mq_monitor.suggest_baseline(
    baseline_dataset=mq_baseline_uri,
    dataset_format=DatasetFormat.csv(header=True),
    output_s3_uri=f"s3://{bucket}/{prefix}/monitoring/mq-baseline-simple",
    problem_type="MulticlassClassification",
    inference_attribute="prediction",
    ground_truth_attribute="ground_truth_label",
    wait=True,
    logs=False,
)

print("✅ Baselining job complete!")

INFO:sagemaker:Creating processing-job with name baseline-suggestion-job-2026-02-15-04-58-46-806


...........................................................!✅ Baselining job complete!


In [19]:
#4.5 Get baseline files
job_desc = mq_monitor.latest_baselining_job.describe()
output_uri = job_desc['ProcessingOutputConfig']['Outputs'][0]['S3Output']['S3Uri']
mq_stats_uri = f"{output_uri}/statistics.json"
mq_constraints_uri = f"{output_uri}/constraints.json"
print("✅ Stats:", mq_stats_uri)
print("✅ Constraints:", mq_constraints_uri)

baseline_job = dq_monitor.latest_baselining_job

print ("Extracting numerical means/stds")

#Stats: Extract numerical means/stds
stats_list = []
for feature in baseline_job.baseline_statistics().body_dict["features"]:
    name = feature['name']
    num_stats = feature.get('numerical_statistics', {})
    common = num_stats.get('common', {})
    stats_list.append({
        'Feature': name,
        'Type': feature['inferred_type'],
        'Mean': round(num_stats.get('mean', 0), 2),
        'Std': round(num_stats.get('std_dev', 0), 2),
        'Min': round(num_stats.get('min', 0), 2),
        'Max': round(num_stats.get('max', 0), 2),
        'Count': common.get('num_present', 0)
    })
stats_df = pd.DataFrame(stats_list)
print("📊 Fixed DQ Stats:")
print(stats_df.to_markdown(index=False, numalign="right", stralign="left"))


#Constraints: Key rules
constraints_list = []
for feature in baseline_job.suggested_constraints().body_dict["features"]:
    name = feature['name']
    constraints_list.append({
        'Feature': name,
        'Completeness': feature['completeness'],
        'Constraints': str(feature.get('num_constraints', {}))
    })
constraints_df = pd.DataFrame(constraints_list)

print("\n📏 DQ Constraints:")
print(constraints_df.to_markdown(index=False, numalign="right", stralign="left"))

✅ Stats: s3://sagemaker-us-east-1-513691803389/models/benchmarks/monitoring/mq-baseline-simple/statistics.json
✅ Constraints: s3://sagemaker-us-east-1-513691803389/models/benchmarks/monitoring/mq-baseline-simple/constraints.json
Extracting numerical means/stds
📊 Fixed DQ Stats:
| Feature   | Type       |    Mean |     Std |      Min |     Max |   Count |
|:----------|:-----------|--------:|--------:|---------:|--------:|--------:|
| meanfreq  | Fractional | 2348.25 | 1468.42 |        0 | 7655.34 |   10242 |
| sd        | Fractional | 2447.62 | 1105.57 |        0 | 6324.35 |   10242 |
| median    | Fractional | 4593.04 | 2696.54 |        0 | 14629.6 |   10242 |
| q25       | Fractional |    0.09 |    0.05 |        0 |    0.32 |   10242 |
| q75       | Fractional | -404.31 |  106.93 | -1131.37 |    0.25 |   10242 |
| iqr       | Fractional |   87.61 |   25.56 |    -4.72 |  169.43 |   10242 |
| skew      | Fractional |   19.79 |    22.2 |   -53.82 |    61.2 |   10242 |
| kurt      | Fract

In [20]:
#4.6 Create a Model Quality Monitoring Schedule

correct_prefix = "models/benchmarks"

mq_schedule_name = "xgb-model-quality-schedule"

mq_monitor.create_monitoring_schedule(
    monitor_schedule_name=mq_schedule_name,
    endpoint_input=EndpointInput(
        endpoint_name=xgb_endpoint_name,
        destination="/opt/ml/processing/input/endpoint",
        inference_attribute="0",
        start_time_offset="-PT1H",  # Look back 1 hour
        end_time_offset="-PT0H",
    ),
    problem_type="MulticlassClassification",
    
    # CHECK FILE PATH HERE
    ground_truth_input=f"s3://{bucket}/{correct_prefix}/ground-truth",
    
    # Output can stay in monitoring subfolder
    output_s3_uri=f"s3://{bucket}/{correct_prefix}/monitoring/model-quality",
    
    schedule_cron_expression=CronExpressionGenerator.hourly(),
    enable_cloudwatch_metrics=True,
)

print(f"✅ Model quality schedule created: {mq_schedule_name}")
print(f"\n📂 Configured paths:")
print(f"   Ground Truth: s3://{bucket}/{correct_prefix}/ground-truth")
print(f"   Data Capture: s3://{bucket}/{correct_prefix}/datacapture (from endpoint)")
print(f"   Output: s3://{bucket}/{correct_prefix}/monitoring/model-quality")

INFO:sagemaker.model_monitor.model_monitoring:Creating Monitoring Schedule with name: xgb-model-quality-schedule


✅ Model quality schedule created: xgb-model-quality-schedule

📂 Configured paths:
   Ground Truth: s3://sagemaker-us-east-1-513691803389/models/benchmarks/ground-truth
   Data Capture: s3://sagemaker-us-east-1-513691803389/models/benchmarks/datacapture (from endpoint)
   Output: s3://sagemaker-us-east-1-513691803389/models/benchmarks/monitoring/model-quality


In [21]:
correct_prefix = "models/benchmarks"
ground_truth_input=f"s3://{bucket}/{correct_prefix}/ground-truth",
# This becomes:
# s3://.../models/benchmarks/ground-truth ✅

In [22]:
#4.7 Mimick traffic to endpoint
def invoke_endpoint_with_traffic(endpoint_name, data_file, labels_file, requests_per_minute, session_obj):
    """
    Simulate production traffic using baseline samples
    """
    try:
        # Load features and labels
        X_df = pd.read_csv(data_file)
        full_df = pd.read_csv(labels_file)
        
        sleep_time = 60.0 / requests_per_minute
        
        print(f"🚀 Starting traffic simulation...")
        print(f"   Endpoint: {endpoint_name}")
        print(f"   Rate: {requests_per_minute} requests/minute")
        print(f"   Samples: {len(X_df)}")
        
        # Create runtime client inside function
        runtime_client = session_obj.sagemaker_runtime_client
        
        i = 0
        request_count = 0
        
        while True:
            try:
                # Cycle through samples
                row_idx = i % len(X_df)
                row_data = X_df.iloc[row_idx]
                
                # Convert to CSV format
                payload = ','.join(map(str, row_data.values))
                
                # Invoke endpoint
                response = runtime_client.invoke_endpoint(
                    EndpointName=endpoint_name,
                    ContentType="text/csv",
                    Accept="text/csv",
                    Body=payload,
                    InferenceId=str(i),
                )
                
                result = response["Body"].read().decode('utf-8')
                request_count += 1
                
                if request_count % 10 == 0:
                    actual_label = full_df.iloc[row_idx]['target']
                    print(f"   Request {request_count}: prediction={result.strip()[:20]}, actual={actual_label}")
                
                i += 1
                sleep(sleep_time)
                
            except Exception as e:
                print(f"⚠️  Error on request {i}: {e}")
                sleep(5)
                
    except Exception as e:
        print(f"❌ Fatal error in traffic simulation: {e}")

# Start simulation
traffic_thread = Thread(
    target=invoke_endpoint_with_traffic,
    args=(xgb_endpoint_name, "test_data/traffic_data.csv", "test_data/traffic_with_labels.csv", 30, session),
    daemon=True
)
traffic_thread.start()
print("✅ Traffic simulation started")

# Wait a bit for some traffic to flow
print("\n⏳ Waiting 30 seconds for traffic to generate...")
sleep(30)

# Check for captured data
s3_capture_upload_path = f"s3://{bucket}/{prefix}/datacapture"

print("\n🔍 Checking for captured inference data...")
print("Waiting for captures to show up", end="")

capture_files = []
capture_file = None
capture_record = None

for i in range(120):
    capture_files = sorted(S3Downloader.list(f"{s3_capture_upload_path}/{xgb_endpoint_name}"))
    if capture_files:
        # Check if the file has inferenceId
        try:
            capture_file = S3Downloader.read_file(capture_files[-1]).split("\n")
            capture_record = json.loads(capture_file[0])
            if "inferenceId" in capture_record["eventMetadata"]:
                print(" ✓")
                break
        except:
            pass
    print(".", end="", flush=True)
    sleep(1)

if not capture_files:
    print(" ❌")
    print("No capture files found. Check:")
    print(f"  - Data capture path: {s3_capture_upload_path}")
    print(f"  - Endpoint: {xgb_endpoint_name}")
else:
    print("\n✅ Found Capture Files:")
    print("\n ".join(capture_files[-3:]))
    
    # 4. VIEW CONTENTS - Show raw lines
    print("\n" + "="*80)
    print("📄 RAW CAPTURE FILE CONTENTS (last 3 lines):")
    print("="*80)
    if capture_file and len(capture_file) >= 3:
        print("\n".join(capture_file[-3:-1]))  # Show last 2 complete lines
    else:
        print("\n".join(capture_file[:2]))  # Show first 2 lines if file is small
    
    # 5. FORMATTED VIEW - Pretty print a single record
    print("\n" + "="*80)
    print("📋 FORMATTED CAPTURE RECORD (single line):")
    print("="*80)
    if capture_record:
        print(json.dumps(capture_record, indent=2))
        
        # 6. VALIDATE inferenceId
        print("\n" + "="*80)
        print("✅ VALIDATION:")
        print("="*80)
        if "inferenceId" in capture_record.get("eventMetadata", {}):
            inference_id = capture_record["eventMetadata"]["inferenceId"]
            print(f"✓ inferenceId found: '{inference_id}'")
            print("  → This will be used to join with ground truth data")
        else:
            event_id = capture_record.get("eventMetadata", {}).get("eventId", "N/A")
            print(f"⚠ No inferenceId found, will use eventId: '{event_id}'")
        
        # Show what was captured
        if "captureData" in capture_record:
            input_data = capture_record["captureData"].get("endpointInput", {})
            output_data = capture_record["captureData"].get("endpointOutput", {})
            
            print(f"\n📥 Input captured: {input_data.get('data', 'N/A')[:50]}...")
            print(f"📤 Output captured: {output_data.get('data', 'N/A')[:50]}...")
            print(f"⏰ Inference time: {capture_record['eventMetadata'].get('inferenceTime', 'N/A')}")

✅ Traffic simulation started

⏳ Waiting 30 seconds for traffic to generate...
🚀 Starting traffic simulation...
   Endpoint: xgb-benchmark-endpoint-20260215-045008
   Rate: 30 requests/minute
   Samples: 300
   Request 10: prediction=0.01371731422841549,, actual=4.0

🔍 Checking for captured inference data...
Waiting for captures to show up.......   Request 20: prediction=0.010298002511262894, actual=1.0
................   Request 30: prediction=0.1026264950633049,0, actual=5.0
......... ✓

✅ Found Capture Files:
s3://sagemaker-us-east-1-513691803389/models/benchmarks/datacapture/xgb-benchmark-endpoint-20260215-045008/AllTraffic/2026/02/15/04/58-46-503-8588dc6d-7194-41aa-9067-54a7050616ff.jsonl
 s3://sagemaker-us-east-1-513691803389/models/benchmarks/datacapture/xgb-benchmark-endpoint-20260215-045008/AllTraffic/2026/02/15/05/03-50-510-c36806db-17af-4f7c-b8cb-40cbce7c5fa8.jsonl

📄 RAW CAPTURE FILE CONTENTS (last 3 lines):
{"captureData":{"endpointInput":{"observedContentType":"text/csv","

In [24]:
#4.8 Upload ground truth for these samples

def upload_baseline_ground_truth(hours_offset=0):
    """
    Upload ground truth
    
    Args:
        hours_offset: Upload for current hour + offset (0=now, -1=last hour, 1=next hour)
    """
    # Read the labeled traffic data
    full_df = pd.read_csv("test_data/traffic_with_labels.csv")
    
    # Calculate target time
    target_time = datetime.now(timezone.utc) + timedelta(hours=hours_offset)
    
    # CRITICAL: Use DIRECTORY structure YYYY/MM/DD/HH/ not timestamp filename
    utc_hour_dir = target_time.strftime("%Y/%m/%d/%H/")
    gt_key = f"models/benchmarks/ground-truth/{utc_hour_dir}ground-truth.jsonl"    
    
    # Create ground truth records with CORRECT format
    ground_truth_data = []
    for i in range(len(full_df)):
        record = {
            "groundTruthData": {  # Note: "groundTruthData" not "groundTruth"
                "data": str(full_df.iloc[i]['target']),
                "encoding": "CSV"
            },
            "eventMetadata": {
                "eventId": str(i)  # Matches inferenceId from traffic simulation
            },
            "eventVersion": "0"
        }
        ground_truth_data.append(json.dumps(record))
    
    # Upload with proper JSONL format (newline-separated)
    s3_client.put_object(
        Bucket=bucket,
        Key=gt_key,
        Body="\n".join(ground_truth_data),
        ContentType='application/jsonlines'
    )
    
    print(f"✅ Uploaded {len(ground_truth_data)} ground truth labels")
    print(f"   Time: {target_time.strftime('%Y-%m-%d %H:00 UTC')}")
    print(f"   Path: s3://{bucket}/{gt_key}")
    
    return gt_key

# Upload for last hour, current hour, and next hour
print("=" * 80)
print("UPLOADING GROUND TRUTH - CORRECT DIRECTORY STRUCTURE")
print("=" * 80)

upload_baseline_ground_truth(-1)  # Last hour (in case monitor already ran)
upload_baseline_ground_truth(0)   # Current hour
upload_baseline_ground_truth(1)   # Next hour

print("\n" + "=" * 80)
print("VERIFICATION")
print("=" * 80)

# Verify files were uploaded
import boto3
s3 = boto3.client('s3')

print("\n📁 Ground Truth Files in S3:")
try:
    response = s3.list_objects_v2(
        Bucket=bucket,
        Prefix=f"{prefix}/ground-truth/"
    )
    
    if 'Contents' in response:
        for obj in response['Contents']:
            key = obj['Key']
            size_kb = obj['Size'] / 1024
            # Extract the hour from the path
            parts = key.split('/')
            if len(parts) >= 6:
                hour_path = '/'.join(parts[-5:-1])  # YYYY/MM/DD/HH
                print(f"   ✅ {hour_path}/ - {size_kb:.1f} KB")
            else:
                print(f"   • {key} - {size_kb:.1f} KB")
    else:
        print("   ❌ No files found!")
except Exception as e:
    print(f"   ❌ Error: {e}")

print("\n" + "=" * 80)

UPLOADING GROUND TRUTH - CORRECT DIRECTORY STRUCTURE
✅ Uploaded 300 ground truth labels
   Time: 2026-02-15 04:00 UTC
   Path: s3://sagemaker-us-east-1-513691803389/models/benchmarks/ground-truth/2026/02/15/04/ground-truth.jsonl
✅ Uploaded 300 ground truth labels
   Time: 2026-02-15 05:00 UTC
   Path: s3://sagemaker-us-east-1-513691803389/models/benchmarks/ground-truth/2026/02/15/05/ground-truth.jsonl
✅ Uploaded 300 ground truth labels
   Time: 2026-02-15 06:00 UTC
   Path: s3://sagemaker-us-east-1-513691803389/models/benchmarks/ground-truth/2026/02/15/06/ground-truth.jsonl

VERIFICATION

📁 Ground Truth Files in S3:
   ✅ 2026/02/12/06/ - 0.8 KB
   ✅ 2026/02/12/07/ - 0.8 KB
   ✅ 2026/02/15/04/ - 33.3 KB
   ✅ 2026/02/15/05/ - 33.3 KB
   ✅ 2026/02/15/06/ - 33.3 KB



In [25]:
#4.8 Check monitor and executions
mq_monitor.describe_schedule()

print ("Prediction Runs: ")
mq_executions = mq_monitor.list_executions()
mq_executions

Prediction Runs: 


   Request 100: prediction=0.010600213892757893, actual=1.0


In [26]:
def quick_check():
    """Quick monitoring status check"""
    from datetime import datetime, timezone
    
    schedules = ["xgb-data-quality-schedule", "xgb-model-quality-schedule"]
    
    print(f"\n⏰ {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC')}")
    print("=" * 60)
    
    for s in schedules:
        try:
            r = sm_client.describe_monitoring_schedule(MonitoringScheduleName=s)
            e = r.get("LastMonitoringExecutionSummary", {})
            status = e.get("MonitoringExecutionStatus", "No run")
            scheduled = e.get("ScheduledTime", "")
            
            if status in ["Completed", "CompletedWithViolations"]:
                icon = "✅"
            elif status == "Failed":
                icon = "❌"
            elif status == "InProgress":
                icon = "⏳"
            else:
                icon = "⏸️"
            
            schedule_short = s.replace("xgb-", "").replace("-schedule", "")
            print(f"{icon} {schedule_short:15}: {status}")
            
        except Exception as e:
            print(f"❌ {s}: Error")
    
    print("=" * 60)

# Run it now
quick_check()


⏰ 2026-02-15 05:07 UTC
✅ data-quality   : CompletedWithViolations
❌ model-quality  : Failed


In [27]:
# Final Pre-Flight Check
from datetime import datetime, timezone, timedelta
import s3fs

print("=" * 80)
print("PRE-FLIGHT CHECK - Will Monitor Succeed?")
print("=" * 80)

fs = s3fs.S3FileSystem()
current_hour = datetime.now(timezone.utc)

# Check the hour the monitor will process next
processing_hour = current_hour.replace(minute=0, second=0, microsecond=0)
hour_dir = processing_hour.strftime("%Y/%m/%d/%H")

print(f"\n⏰ Current Time: {current_hour.strftime('%Y-%m-%d %H:%M UTC')}")
print(f"📅 Next Monitor Run: {(processing_hour + timedelta(hours=1)).strftime('%Y-%m-%d %H:00 UTC')}")
print(f"📂 Will Process Hour: {hour_dir}")

# 1. Check Data Capture exists
print(f"\n1️⃣  DATA CAPTURE CHECK:")
dc_path = f"{bucket}/{prefix.rstrip('/')}/datacapture/{xgb_endpoint_name}/AllTraffic/{hour_dir}"
try:
    dc_files = fs.ls(dc_path)
    dc_size = sum(fs.size(f) for f in dc_files) / 1024
    print(f"   ✅ {len(dc_files)} files ({dc_size:.1f} KB)")
    print(f"   Path: s3://{dc_path}")
except Exception as e:
    print(f"   ❌ No data capture files!")
    print(f"   Path: s3://{dc_path}")

# 2. Check Ground Truth exists
print(f"\n2️⃣  GROUND TRUTH CHECK:")
gt_path = f"{bucket}/models/benchmarks/ground-truth/{hour_dir}"
try:
    gt_files = fs.ls(gt_path)
    if gt_files:
        gt_size = sum(fs.size(f) for f in gt_files) / 1024
        print(f"   ✅ {len(gt_files)} file(s) ({gt_size:.1f} KB)")
        print(f"   Path: s3://{gt_path}")
    else:
        print(f"   ❌ No ground truth files!")
except Exception as e:
    print(f"   ❌ No ground truth files!")
    print(f"   Path: s3://{gt_path}")

# 3. Check Monitor Schedule Status
print(f"\n3️⃣  MONITOR STATUS:")
try:
    r = sm_client.describe_monitoring_schedule(MonitoringScheduleName="xgb-model-quality-schedule")
    schedule_status = r['MonitoringScheduleStatus']
    print(f"   Schedule: {schedule_status}")
    
    if schedule_status != "Scheduled":
        print(f"   ⚠️  WARNING: Schedule is not active!")
except Exception as e:
    print(f"   ❌ Error: {e}")

# 4. Check Baseline exists
print(f"\n4️⃣  BASELINE CHECK:")
baseline_path = f"{bucket}/models/benchmarks/mq_baseline.csv"
try:
    if fs.exists(baseline_path):
        print(f"   ✅ Baseline exists")
    else:
        print(f"   ❌ Baseline missing!")
except:
    print(f"   ❌ Baseline missing!")

# Final verdict
print(f"\n" + "=" * 80)
print("VERDICT")
print("=" * 80)

checks = {
    "Data Capture": "✅",
    "Ground Truth": "✅",
    "Monitor Active": "✅",
    "Baseline": "✅"
}

all_good = True
for check, status in checks.items():
    print(f"{status} {check}")
    if status == "❌":
        all_good = False

if all_good:
    print(f"\n🎉 ALL CHECKS PASSED!")
    print(f"✅ Model Quality monitor SHOULD succeed at next run")
    print(f"⏰ Next execution: {(processing_hour + timedelta(hours=1)).strftime('%H:%M UTC')}")
else:
    print(f"\n⚠️  SOME CHECKS FAILED - Monitor may fail")

print("=" * 80)

PRE-FLIGHT CHECK - Will Monitor Succeed?

⏰ Current Time: 2026-02-15 05:07 UTC
📅 Next Monitor Run: 2026-02-15 06:00 UTC
📂 Will Process Hour: 2026/02/15/05

1️⃣  DATA CAPTURE CHECK:
   ✅ 3 files (67.6 KB)
   Path: s3://sagemaker-us-east-1-513691803389/models/benchmarks/datacapture/xgb-benchmark-endpoint-20260215-045008/AllTraffic/2026/02/15/05

2️⃣  GROUND TRUTH CHECK:
   ✅ 1 file(s) (33.3 KB)
   Path: s3://sagemaker-us-east-1-513691803389/models/benchmarks/ground-truth/2026/02/15/05

3️⃣  MONITOR STATUS:
   Schedule: Scheduled

4️⃣  BASELINE CHECK:
   ✅ Baseline exists

VERDICT
✅ Data Capture
✅ Ground Truth
✅ Monitor Active
✅ Baseline

🎉 ALL CHECKS PASSED!
✅ Model Quality monitor SHOULD succeed at next run
⏰ Next execution: 06:00 UTC
   Request 110: prediction=0.05263245105743408,, actual=0.0
   Request 120: prediction=0.022993164137005806, actual=1.0


## 6. Infrastructure Monitors (CloudWatch Alarms)

In [28]:
#6.1 Create CloudWatch Alarms for All Monitors
cw_client = boto3.client('cloudwatch', region_name=region)
account_id = boto3.client('sts').get_caller_identity()['Account']

print("🚨 Creating CloudWatch Alarms...")
print("=" * 80)

# Common dimensions
endpoint_metric_dimensions = [
    {"Name": "EndpointName", "Value": xgb_endpoint_name},
]

model_quality_dimensions = [
    {"Name": "Endpoint", "Value": xgb_endpoint_name},
    {"Name": "MonitoringSchedule", "Value": "xgb-model-quality-schedule"}
]

data_quality_dimensions = [
    {"Name": "Endpoint", "Value": xgb_endpoint_name},
    {"Name": "MonitoringSchedule", "Value": "xgb-data-quality-schedule"}
]

alarms_created = []

# ============================================================================
# INFRASTRUCTURE ALARMS
# ============================================================================
print("\n📊 Creating Infrastructure Alarms...")

# 1. High Latency
cw_client.put_metric_alarm(
    AlarmName="XGB-Endpoint-High-Latency",
    AlarmDescription="Model latency above 10 seconds",
    Namespace="AWS/SageMaker",
    MetricName="ModelLatency",
    Dimensions=endpoint_metric_dimensions,
    Statistic="Average",
    Period=300,
    EvaluationPeriods=2,
    Threshold=10000.0,  # 10 seconds in milliseconds
    ComparisonOperator="GreaterThanThreshold",
    TreatMissingData="notBreaching",
    ActionsEnabled=False,
)
alarms_created.append("XGB-Endpoint-High-Latency")
print("  ✓ High Latency alarm created")

# 2. Invocation Errors
cw_client.put_metric_alarm(
    AlarmName="XGB-Endpoint-Invocation-Errors",
    AlarmDescription="Invocation errors for endpoint",
    Namespace="AWS/SageMaker",
    MetricName="ModelInvocationErrors",
    Dimensions=endpoint_metric_dimensions,
    Statistic="Sum",
    Period=300,
    EvaluationPeriods=1,
    Threshold=5.0,
    ComparisonOperator="GreaterThanThreshold",
    TreatMissingData="notBreaching",
    ActionsEnabled=False,
)
alarms_created.append("XGB-Endpoint-Invocation-Errors")
print("  ✓ Invocation Errors alarm created")

# 3. 5XX Errors
cw_client.put_metric_alarm(
    AlarmName="XGB-Endpoint-5XX-Errors-High",
    AlarmDescription="5XX error rate high",
    Namespace="AWS/SageMaker",
    MetricName="Invocation5XXErrors",  # Fixed metric name
    Dimensions=endpoint_metric_dimensions,
    Statistic="Sum",
    Period=300,
    EvaluationPeriods=2,
    Threshold=5.0,
    ComparisonOperator="GreaterThanThreshold",
    TreatMissingData="notBreaching",
    ActionsEnabled=False,
)
alarms_created.append("XGB-Endpoint-5XX-Errors-High")
print("  ✓ 5XX Errors alarm created")

# 4. 4XX Errors
cw_client.put_metric_alarm(
    AlarmName="XGB-Endpoint-4XX-Errors-High",
    AlarmDescription="4XX error rate high",
    Namespace="AWS/SageMaker",
    MetricName="Invocation4XXErrors",  # Fixed metric name
    Dimensions=endpoint_metric_dimensions,
    Statistic="Sum",
    Period=300,
    EvaluationPeriods=2,
    Threshold=10.0,
    ComparisonOperator="GreaterThanThreshold",
    TreatMissingData="notBreaching",
    ActionsEnabled=False,
)
alarms_created.append("XGB-Endpoint-4XX-Errors-High")
print("  ✓ 4XX Errors alarm created")

# 5. High CPU Utilization
cw_client.put_metric_alarm(
    AlarmName="XGB-Endpoint-High-CPU",
    AlarmDescription="CPU utilization above 80%",
    Namespace="AWS/SageMaker",
    MetricName="CPUUtilization",
    Dimensions=endpoint_metric_dimensions + [{"Name": "VariantName", "Value": "AllTraffic"}],
    Statistic="Average",
    Period=300,
    EvaluationPeriods=2,
    Threshold=80.0,
    ComparisonOperator="GreaterThanThreshold",
    TreatMissingData="notBreaching",
    ActionsEnabled=False,
)
alarms_created.append("XGB-Endpoint-High-CPU")
print("  ✓ High CPU alarm created")

# 6. High Memory Utilization
cw_client.put_metric_alarm(
    AlarmName="XGB-Endpoint-High-Memory",
    AlarmDescription="Memory utilization above 85%",
    Namespace="AWS/SageMaker",
    MetricName="MemoryUtilization",
    Dimensions=endpoint_metric_dimensions + [{"Name": "VariantName", "Value": "AllTraffic"}],
    Statistic="Average",
    Period=300,
    EvaluationPeriods=2,
    Threshold=85.0,
    ComparisonOperator="GreaterThanThreshold",
    TreatMissingData="notBreaching",
    ActionsEnabled=False,
)
alarms_created.append("XGB-Endpoint-High-Memory")
print("  ✓ High Memory alarm created")

# ============================================================================
# MODEL QUALITY ALARMS
# ============================================================================
print("\n🎯 Creating Model Quality Alarms...")

# 7. Low Accuracy
cw_client.put_metric_alarm(
    AlarmName="XGB-Accuracy-Low",
    AlarmDescription="Accuracy below baseline threshold",
    ActionsEnabled=False,
    MetricName="accuracy",
    Namespace="aws/sagemaker/Endpoints/model-metrics",
    Statistic="Average",
    Dimensions=model_quality_dimensions,
    Period=3600,  # 1 hour
    EvaluationPeriods=1,
    DatapointsToAlarm=1,
    Threshold=0.70,  # Adjust based on your baseline
    ComparisonOperator="LessThanOrEqualToThreshold",
    TreatMissingData="notBreaching"
)
alarms_created.append("XGB-Accuracy-Low")
print("  ✓ Low Accuracy alarm created")

# 8. Low F1 Score
cw_client.put_metric_alarm(
    AlarmName="XGB-F1-Score-Low",
    AlarmDescription="F1 score below baseline threshold",
    ActionsEnabled=False,
    MetricName="f1",
    Namespace="aws/sagemaker/Endpoints/model-metrics",
    Statistic="Average",
    Dimensions=model_quality_dimensions,
    Period=3600,
    EvaluationPeriods=1,
    DatapointsToAlarm=1,
    Threshold=0.65,  # Adjust based on your baseline
    ComparisonOperator="LessThanOrEqualToThreshold",
    TreatMissingData="notBreaching"
)
alarms_created.append("XGB-F1-Score-Low")
print("  ✓ Low F1 Score alarm created")

# 9. Low Precision
cw_client.put_metric_alarm(
    AlarmName="XGB-Precision-Low",
    AlarmDescription="Precision below baseline threshold",
    ActionsEnabled=False,
    MetricName="precision",
    Namespace="aws/sagemaker/Endpoints/model-metrics",
    Statistic="Average",
    Dimensions=model_quality_dimensions,
    Period=3600,
    EvaluationPeriods=1,
    DatapointsToAlarm=1,
    Threshold=0.60,  # Adjust based on your baseline
    ComparisonOperator="LessThanOrEqualToThreshold",
    TreatMissingData="notBreaching"
)
alarms_created.append("XGB-Precision-Low")
print("  ✓ Low Precision alarm created")

# 10. Low Recall
cw_client.put_metric_alarm(
    AlarmName="XGB-Recall-Low",
    AlarmDescription="Recall below baseline threshold",
    ActionsEnabled=False,
    MetricName="recall",
    Namespace="aws/sagemaker/Endpoints/model-metrics",
    Statistic="Average",
    Dimensions=model_quality_dimensions,
    Period=3600,
    EvaluationPeriods=1,
    DatapointsToAlarm=1,
    Threshold=0.60,  # Adjust based on your baseline
    ComparisonOperator="LessThanOrEqualToThreshold",
    TreatMissingData="notBreaching"
)
alarms_created.append("XGB-Recall-Low")
print("  ✓ Low Recall alarm created")

# ============================================================================
# DATA QUALITY ALARMS
# ============================================================================
print("\n📉 Creating Data Quality Alarms...")

# 11. Feature Drift
cw_client.put_metric_alarm(
    AlarmName="XGB-Feature-Drift-High",
    AlarmDescription="Feature drift detected",
    ActionsEnabled=False,
    MetricName="feature_baseline_drift_total_amount",
    Namespace="aws/sagemaker/Endpoints/data-metrics",
    Statistic="Average",
    Dimensions=data_quality_dimensions,
    Period=3600,
    EvaluationPeriods=1,
    DatapointsToAlarm=1,
    Threshold=0.5,  # Adjust based on acceptable drift
    ComparisonOperator="GreaterThanThreshold",
    TreatMissingData="notBreaching"
)
alarms_created.append("XGB-Feature-Drift-High")
print("  ✓ Feature Drift alarm created")

# 12. Data Completeness Issues
cw_client.put_metric_alarm(
    AlarmName="XGB-Data-Completeness-Low",
    AlarmDescription="Data completeness below threshold",
    ActionsEnabled=False,
    MetricName="completeness_baseline_drift_total_amount",
    Namespace="aws/sagemaker/Endpoints/data-metrics",
    Statistic="Average",
    Dimensions=data_quality_dimensions,
    Period=3600,
    EvaluationPeriods=1,
    DatapointsToAlarm=1,
    Threshold=0.3,  # Adjust based on acceptable missing data
    ComparisonOperator="GreaterThanThreshold",
    TreatMissingData="notBreaching"
)
alarms_created.append("XGB-Data-Completeness-Low")
print("  ✓ Data Completeness alarm created")

# ============================================================================
# SUMMARY
# ============================================================================
print("\n" + "=" * 80)
print("✅ ALL ALARMS CREATED SUCCESSFULLY")
print("=" * 80)
print(f"\nTotal Alarms Created: {len(alarms_created)}")
print("\nInfrastructure Alarms (6):")
print("  • High Latency")
print("  • Invocation Errors")
print("  • 5XX Errors")
print("  • 4XX Errors")
print("  • High CPU")
print("  • High Memory")

print("\nModel Quality Alarms (4):")
print("  • Low Accuracy")
print("  • Low F1 Score")
print("  • Low Precision")
print("  • Low Recall")

print("\nData Quality Alarms (2):")
print("  • High Feature Drift")
print("  • Low Data Completeness")

print("\n🔗 View All Alarms:")
alarms_url = f"https://{region}.console.aws.amazon.com/cloudwatch/home?region={region}#alarmsV2:"
print(alarms_url)

print("\n⚠️  Note: Model and Data Quality alarms won't trigger until monitoring jobs run (1+ hours)")
print("=" * 80)

🚨 Creating CloudWatch Alarms...

📊 Creating Infrastructure Alarms...
  ✓ High Latency alarm created
  ✓ Invocation Errors alarm created
  ✓ 5XX Errors alarm created
  ✓ 4XX Errors alarm created
  ✓ High CPU alarm created
  ✓ High Memory alarm created

🎯 Creating Model Quality Alarms...
  ✓ Low Accuracy alarm created
  ✓ Low F1 Score alarm created
  ✓ Low Precision alarm created
  ✓ Low Recall alarm created

📉 Creating Data Quality Alarms...
  ✓ Feature Drift alarm created
  ✓ Data Completeness alarm created

✅ ALL ALARMS CREATED SUCCESSFULLY

Total Alarms Created: 12

Infrastructure Alarms (6):
  • High Latency
  • Invocation Errors
  • 5XX Errors
  • 4XX Errors
  • High CPU
  • High Memory

Model Quality Alarms (4):
  • Low Accuracy
  • Low F1 Score
  • Low Precision
  • Low Recall

Data Quality Alarms (2):
  • High Feature Drift
  • Low Data Completeness

🔗 View All Alarms:
https://us-east-1.console.aws.amazon.com/cloudwatch/home?region=us-east-1#alarmsV2:

⚠️  Note: Model and Data Q

## 7. CloudWatch Monitoring Dashboard

In [29]:
import json

#7.1 CloudWatch Dashboard for All ML Monitors
dashboard_name = "SageMaker-ML-Benchmarks-Complete"

dashboard_body = {
    "widgets": [
        # Row 1: Endpoint Performance
        {
            "type": "metric",
            "x": 0,
            "y": 0,
            "width": 8,
            "height": 6,
            "properties": {
                "title": "Endpoint Invocations",
                "metrics": [
                    ["AWS/SageMaker", "Invocations", "EndpointName", xgb_endpoint_name, "VariantName", "AllTraffic"],
                ],
                "view": "timeSeries",
                "stacked": False,
                "region": region,
                "period": 300,
                "stat": "Sum",
                "yAxis": {"left": {"label": "Count"}},
            },
        },
        {
            "type": "metric",
            "x": 8,
            "y": 0,
            "width": 8,
            "height": 6,
            "properties": {
                "title": "Model Latency",
                "metrics": [
                    ["AWS/SageMaker", "ModelLatency", "EndpointName", xgb_endpoint_name, "VariantName", "AllTraffic", {"stat": "Average"}],
                    ["...", {"stat": "p99"}],
                ],
                "view": "timeSeries",
                "stacked": False,
                "region": region,
                "period": 300,
                "yAxis": {"left": {"label": "Milliseconds"}},
            },
        },
        {
            "type": "metric",
            "x": 16,
            "y": 0,
            "width": 8,
            "height": 6,
            "properties": {
                "title": "Endpoint Errors",
                "metrics": [
                    ["AWS/SageMaker", "InvocationModelErrors", "EndpointName", xgb_endpoint_name, "VariantName", "AllTraffic"],
                    [".", "Invocation4XXErrors", ".", ".", ".", "."],
                    [".", "Invocation5XXErrors", ".", ".", ".", "."],
                ],
                "view": "timeSeries",
                "stacked": True,
                "region": region,
                "period": 300,
                "stat": "Sum",
                "yAxis": {"left": {"label": "Count"}},
            },
        },
        
        # Row 2: Infrastructure Metrics
        {
            "type": "metric",
            "x": 0,
            "y": 6,
            "width": 8,
            "height": 6,
            "properties": {
                "title": "CPU Utilization",
                "metrics": [
                    ["AWS/SageMaker", "CPUUtilization", "EndpointName", xgb_endpoint_name, "VariantName", "AllTraffic"],
                ],
                "view": "timeSeries",
                "stacked": False,
                "region": region,
                "period": 300,
                "stat": "Average",
                "yAxis": {"left": {"min": 0, "max": 100, "label": "Percent"}},
            },
        },
        {
            "type": "metric",
            "x": 8,
            "y": 6,
            "width": 8,
            "height": 6,
            "properties": {
                "title": "Memory Utilization",
                "metrics": [
                    ["AWS/SageMaker", "MemoryUtilization", "EndpointName", xgb_endpoint_name, "VariantName", "AllTraffic"],
                ],
                "view": "timeSeries",
                "stacked": False,
                "region": region,
                "period": 300,
                "stat": "Average",
                "yAxis": {"left": {"min": 0, "max": 100, "label": "Percent"}},
            },
        },
        {
            "type": "metric",
            "x": 16,
            "y": 6,
            "width": 8,
            "height": 6,
            "properties": {
                "title": "Disk Utilization",
                "metrics": [
                    ["AWS/SageMaker", "DiskUtilization", "EndpointName", xgb_endpoint_name, "VariantName", "AllTraffic"],
                ],
                "view": "timeSeries",
                "stacked": False,
                "region": region,
                "period": 300,
                "stat": "Average",
                "yAxis": {"left": {"min": 0, "max": 100, "label": "Percent"}},
            },
        },
        
        # Row 3: Model Quality Metrics (will populate after monitoring runs)
        {
            "type": "metric",
            "x": 0,
            "y": 12,
            "width": 12,
            "height": 6,
            "properties": {
                "title": "Model Quality - Accuracy (Appears after 1st monitoring run)",
                "metrics": [
                    ["aws/sagemaker/Endpoints/model-metrics", "accuracy", 
                     "Endpoint", xgb_endpoint_name, 
                     "MonitoringSchedule", "xgb-model-quality-schedule"],
                ],
                "view": "timeSeries",
                "stacked": False,
                "region": region,
                "period": 3600,
                "stat": "Average",
                "yAxis": {"left": {"min": 0, "max": 1, "label": "Accuracy"}},
            },
        },
        {
            "type": "metric",
            "x": 12,
            "y": 12,
            "width": 12,
            "height": 6,
            "properties": {
                "title": "Model Quality - F1 Score (Appears after 1st monitoring run)",
                "metrics": [
                    ["aws/sagemaker/Endpoints/model-metrics", "f1", 
                     "Endpoint", xgb_endpoint_name, 
                     "MonitoringSchedule", "xgb-model-quality-schedule"],
                ],
                "view": "timeSeries",
                "stacked": False,
                "region": region,
                "period": 3600,
                "stat": "Average",
                "yAxis": {"left": {"min": 0, "max": 1, "label": "F1 Score"}},
            },
        },
        
        # Row 4: Data Quality Metrics (will populate after monitoring runs)
        {
            "type": "metric",
            "x": 0,
            "y": 18,
            "width": 12,
            "height": 6,
            "properties": {
                "title": "Data Quality - Feature Drift (Appears after 1st monitoring run)",
                "metrics": [
                    ["aws/sagemaker/Endpoints/data-metrics", "feature_baseline_drift_total_amount",
                     "Endpoint", xgb_endpoint_name,
                     "MonitoringSchedule", "xgb-data-quality-schedule"],
                ],
                "view": "timeSeries",
                "stacked": False,
                "region": region,
                "period": 3600,
                "stat": "Average",
                "yAxis": {"left": {"label": "Drift Amount"}},
            },
        },
        {
            "type": "metric",
            "x": 12,
            "y": 18,
            "width": 12,
            "height": 6,
            "properties": {
                "title": "Data Quality - Completeness (Appears after 1st monitoring run)",
                "metrics": [
                    ["aws/sagemaker/Endpoints/data-metrics", "completeness_baseline_drift_total_amount",
                     "Endpoint", xgb_endpoint_name,
                     "MonitoringSchedule", "xgb-data-quality-schedule"],
                ],
                "view": "timeSeries",
                "stacked": False,
                "region": region,
                "period": 3600,
                "stat": "Average",
                "yAxis": {"left": {"label": "Drift Amount"}},
            },
        },
        
        # Row 5: Alarms
        {
            "type": "metric",
            "x": 0,
            "y": 24,
            "width": 24,
            "height": 6,
            "properties": {
                "title": "Alarm States - All Monitors",
                "annotations": {
                    "horizontal": [
                        {"value": 0, "label": "OK", "color": "#2ca02c"},
                        {"value": 1, "label": "ALARM", "color": "#d62728"},
                    ]
                },
                "metrics": [
                    ["AWS/CloudWatch", "AlarmState", "AlarmName", "XGB-Endpoint-High-Latency"],
                    ["...", "XGB-Endpoint-Invocation-Errors"],
                    ["...", "XGB-Endpoint-5XX-Errors-High"],
                    ["...", "XGB-Endpoint-4XX-Errors-High"],
                    ["...", "XGB-Endpoint-High-CPU"],
                    ["...", "XGB-Endpoint-High-Memory"],
                    ["...", "XGB-Accuracy-Low"],
                    ["...", "XGB-F1-Score-Low"],
                    ["...", "XGB-Precision-Low"],
                    ["...", "XGB-Recall-Low"],
                    ["...", "XGB-Feature-Drift-High"],
                    ["...", "XGB-Data-Completeness-Low"],
                ],
                "view": "timeSeries",
                "stacked": False,
                "region": region,
                "period": 300,
                "stat": "Maximum",
                "yAxis": {"left": {"min": 0, "max": 1}},
            },
        },
    ]
}

# Create the dashboard
try:
    cw_client.put_dashboard(
        DashboardName=dashboard_name,
        DashboardBody=json.dumps(dashboard_body),
    )
    
    print("\n" + "=" * 80)
    print("✅ DASHBOARD CREATED SUCCESSFULLY")
    print("=" * 80)
    print(f"Dashboard Name: {dashboard_name}")
    print(f"Endpoint: {xgb_endpoint_name}")
    print(f"\n🔗 View Dashboard:")
    print(f"https://console.aws.amazon.com/cloudwatch/home?region={region}#dashboards:name={dashboard_name}")
    print("\n📊 Dashboard includes:")
    print("  ✓ Endpoint Performance (Invocations, Latency, Errors)")
    print("  ✓ Infrastructure Metrics (CPU, Memory, Disk)")
    print("  ✓ Model Quality Metrics (Accuracy, F1 - will populate after monitoring runs)")
    print("  ✓ Data Quality Metrics (Feature Drift, Completeness - will populate after monitoring runs)")
    print("  ✓ Alarm States (All 12 alarms)")
    print("\n⚠️  Note: Model and Data Quality metrics will appear after monitoring jobs complete")
    print("=" * 80)
    
except Exception as e:
    print(f"❌ Error creating dashboard: {e}")


✅ DASHBOARD CREATED SUCCESSFULLY
Dashboard Name: SageMaker-ML-Benchmarks-Complete
Endpoint: xgb-benchmark-endpoint-20260215-045008

🔗 View Dashboard:
https://console.aws.amazon.com/cloudwatch/home?region=us-east-1#dashboards:name=SageMaker-ML-Benchmarks-Complete

📊 Dashboard includes:
  ✓ Endpoint Performance (Invocations, Latency, Errors)
  ✓ Infrastructure Metrics (CPU, Memory, Disk)
  ✓ Model Quality Metrics (Accuracy, F1 - will populate after monitoring runs)
  ✓ Data Quality Metrics (Feature Drift, Completeness - will populate after monitoring runs)
  ✓ Alarm States (All 12 alarms)

⚠️  Note: Model and Data Quality metrics will appear after monitoring jobs complete


## 8. Verification

In [30]:
#8.1 Wait for monitor to run at Top of Hour

def monitor_until_complete():
    """Check every 10 minutes until both schedules complete"""
    schedules = ["xgb-data-quality-schedule", "xgb-model-quality-schedule"]
    
    print("🔄 Monitoring watch started - checking every 10 minutes")
    print("   Press Ctrl+C to stop\n")
    
    check_count = 0
    
    while True:
        check_count += 1
        now = datetime.now(timezone.utc).strftime('%H:%M UTC')
        print(f"\n[Check #{check_count}] {now}")
        print("-" * 60)
        
        all_done = True
        
        for s in schedules:
            try:
                r = sm_client.describe_monitoring_schedule(MonitoringScheduleName=s)
                e = r.get("LastMonitoringExecutionSummary", {})
                status = e.get("MonitoringExecutionStatus", "No run")
                
                if status in ["Completed", "CompletedWithViolations", "Failed"]:
                    icon = "✅" if status.startswith("Completed") else "❌"
                else:
                    icon = "⏳"
                    all_done = False
                
                print(f"{icon} {s}: {status}")
                
            except Exception as e:
                print(f"❌ {s}: Error")
                all_done = False
        
        if all_done:
            print("\n🎉 Both schedules completed!")
            break
        
        print(f"\n⏰ Waiting 5 minutes... (next check at {datetime.now(timezone.utc).timestamp() + 300})")
        time.sleep(300)  # 5 minutes

# Run it
monitor_until_complete()

🔄 Monitoring watch started - checking every 10 minutes
   Press Ctrl+C to stop


[Check #1] 05:08 UTC
------------------------------------------------------------
✅ xgb-data-quality-schedule: CompletedWithViolations
❌ xgb-model-quality-schedule: Failed

🎉 Both schedules completed!


## 9. Generate Model & Data Reports

This is for after the monitor runs.

In [32]:
#Helper to inspect latest executions and get report locations

def get_latest_execution(schedule_name):
    resp = sm_client.list_monitoring_executions(
        MonitoringScheduleName=schedule_name,
        MaxResults=5,
        SortOrder="Descending",
    )
    if not resp["MonitoringExecutionSummaries"]:
        print("No executions found for", schedule_name)
        return None
    return resp["MonitoringExecutionSummaries"][0]

for name in [schedule_name_xgb_dq, "xgb-model-quality-schedule"]:
    latest = get_latest_execution(name)
    if latest:
        print("\nSchedule:", name)
        print(" Status:", latest["MonitoringExecutionStatus"])
        print(" ScheduledTime:", latest["ScheduledTime"])



Schedule: xgb-data-quality-schedule
 Status: CompletedWithViolations
 ScheduledTime: 2026-02-15 05:00:00+00:00

Schedule: xgb-model-quality-schedule
 Status: Failed
 ScheduledTime: 2026-02-15 04:00:00+00:00


In [33]:
#Download latest data quality report artifacts

dq_latest = get_latest_execution(schedule_name_xgb_dq)
if dq_latest and "ProcessingJobArn" in dq_latest:
    job_name = dq_latest["ProcessingJobArn"].split("/")[-1]
    job_desc = sm_client.describe_processing_job(ProcessingJobName=job_name)
    outputs = job_desc["ProcessingOutputConfig"]["Outputs"]
    for out in outputs:
        uri = out["S3Output"]["S3Uri"]
        print("DQ output:", uri)
        # Typically contains /constraints.json and /statistics.json


DQ output: s3://sagemaker-us-east-1-513691803389/models/benchmarks/monitoring/data-quality/xgb/xgb-benchmark-endpoint-20260215-045008/xgb-data-quality-schedule/2026/02/15/05


In [34]:
#Example: download latest model quality report artifacts

mq_latest = get_latest_execution("xgb-model-quality-schedule")
if mq_latest and "ProcessingJobArn" in mq_latest:
    job_name = mq_latest["ProcessingJobArn"].split("/")[-1]
    job_desc = sm_client.describe_processing_job(ProcessingJobName=job_name)
    outputs = job_desc["ProcessingOutputConfig"]["Outputs"]
    for out in outputs:
        uri = out["S3Output"]["S3Uri"]
        print("MQ output:", uri)


   Request 150: prediction=0.03043915517628193,, actual=3.0
   Request 160: prediction=0.06009938567876816,, actual=0.0
   Request 170: prediction=0.037454571574926376, actual=0.0
   Request 180: prediction=0.14618338644504547,, actual=0.0
   Request 190: prediction=0.0729842409491539,0, actual=3.0


## 10. Clean Up Resources

In [37]:
#10.1 Complete Cleanup - Delete ALL Resources to Minimize AWS Spend

sm_client = boto3.client("sagemaker", region_name=region)
cw_client = boto3.client('cloudwatch', region_name=region)
s3_client = boto3.client('s3', region_name=region)

print("=" * 80)
print("AWS RESOURCE CLEANUP - Minimizing Costs")
print("=" * 80)

# ============================================================================
# 1. STOP MONITORING SCHEDULES (before deleting)
# ============================================================================
print("\n📋 Step 1: Stopping Monitoring Schedules...")
schedules = ["xgb-data-quality-schedule", "xgb-model-quality-schedule"]

for schedule in schedules:
    try:
        sm_client.stop_monitoring_schedule(MonitoringScheduleName=schedule)
        print(f"  ⏸️  Stopped: {schedule}")
    except Exception as e:
        print(f"  ⚠️  {schedule}: {e}")

time.sleep(5)

# ============================================================================
# 2. DELETE MONITORING SCHEDULES
# ============================================================================
print("\n🗑️  Step 2: Deleting Monitoring Schedules...")
for schedule in schedules:
    try:
        sm_client.delete_monitoring_schedule(MonitoringScheduleName=schedule)
        print(f"  ✅ Deleted: {schedule}")
    except Exception as e:
        print(f"  ⚠️  {schedule}: {e}")

# ============================================================================
# 3. DELETE ENDPOINT (STOPS BILLING FOR INSTANCE)
# ============================================================================
print("\n🎯 Step 3: Deleting Endpoint (Stops Instance Billing)...")
try:
    sm_client.delete_endpoint(EndpointName=xgb_endpoint_name)
    print(f"  ✅ Deleting endpoint: {xgb_endpoint_name}")
    print(f"     💰 This stops billing for ml.m5.large instance (~$0.115/hour)")
except Exception as e:
    print(f"  ⚠️  Endpoint: {e}")

# ============================================================================
# 4. DELETE ENDPOINT CONFIGURATION
# ============================================================================
print("\n⚙️  Step 4: Deleting Endpoint Configuration...")
try:
    # Get endpoint config name (usually same as endpoint or model name)
    endpoint_config_name = f"{xgb_endpoint_name}-config"
    
    # Try to find the actual config name
    try:
        endpoint_desc = sm_client.describe_endpoint(EndpointName=xgb_endpoint_name)
        endpoint_config_name = endpoint_desc['EndpointConfigName']
    except:
        pass
    
    sm_client.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
    print(f"  ✅ Deleted endpoint config: {endpoint_config_name}")
except Exception as e:
    print(f"  ⚠️  Endpoint config: {e}")

# ============================================================================
# 5. DELETE MODEL (OPTIONAL - doesn't cost money but good for cleanup)
# ============================================================================
print("\n🤖 Step 5: Deleting Model...")
try:
    # Model name is usually related to endpoint name
    model_name = xgb_endpoint_name.replace("-endpoint", "-model")
    
    # List all models and find ours
    models = sm_client.list_models()['Models']
    xgb_models = [m['ModelName'] for m in models if 'xgb' in m['ModelName'].lower()]
    
    for model in xgb_models:
        try:
            sm_client.delete_model(ModelName=model)
            print(f"  ✅ Deleted model: {model}")
        except Exception as e:
            print(f"  ⚠️  {model}: {e}")
            
except Exception as e:
    print(f"  ⚠️  Model deletion: {e}")

# ============================================================================
# 6. DELETE CLOUDWATCH ALARMS (small cost)
# ============================================================================
print("\n⏰ Step 6: Deleting CloudWatch Alarms...")
try:
    alarms = cw_client.describe_alarms(AlarmNamePrefix="XGB-")["MetricAlarms"]
    alarm_names = [a["AlarmName"] for a in alarms]
    
    if alarm_names:
        cw_client.delete_alarms(AlarmNames=alarm_names)
        print(f"  ✅ Deleted {len(alarm_names)} alarms:")
        for name in alarm_names:
            print(f"     - {name}")
    else:
        print(f"  ✅ No XGB alarms found")
except Exception as e:
    print(f"  ⚠️  Alarms: {e}")

# ============================================================================
# 7. DELETE CLOUDWATCH DASHBOARD (no cost, but cleanup)
# ============================================================================
print("\n📊 Step 7: Deleting CloudWatch Dashboard...")
try:
    dashboard_name = "SageMaker-ML-Benchmarks-Complete"
    cw_client.delete_dashboards(DashboardNames=[dashboard_name])
    print(f"  ✅ Deleted dashboard: {dashboard_name}")
except Exception as e:
    print(f"  ⚠️  Dashboard: {e}")

# ============================================================================
# 8. DELETE S3 DATA (COSTS MONEY FOR STORAGE)
# ============================================================================
print("\n💾 Step 8: Cleaning S3 Data (Optional - Comment out to keep data)...")
print("  ⚠️  WARNING: This will delete all captured data and monitoring results!")
print("  ⏸️  Skipping S3 cleanup by default - uncomment below to delete")

# UNCOMMENT THESE LINES TO DELETE S3 DATA:
"""
s3_paths_to_clean = [
    f"{prefix}/datacapture/",
    f"{prefix}/ground-truth/",
    f"{prefix}/monitoring/",
    f"{prefix}/mq_baseline.csv",
]

for s3_path in s3_paths_to_clean:
    try:
        # List and delete all objects with this prefix
        paginator = s3_client.get_paginator('list_objects_v2')
        pages = paginator.paginate(Bucket=bucket, Prefix=s3_path)
        
        objects_to_delete = []
        for page in pages:
            if 'Contents' in page:
                objects_to_delete.extend([{'Key': obj['Key']} for obj in page['Contents']])
        
        if objects_to_delete:
            # Delete in batches of 1000
            for i in range(0, len(objects_to_delete), 1000):
                batch = objects_to_delete[i:i+1000]
                s3_client.delete_objects(
                    Bucket=bucket,
                    Delete={'Objects': batch}
                )
            print(f"  ✅ Deleted {len(objects_to_delete)} objects from s3://{bucket}/{s3_path}")
        else:
            print(f"  ⏭️  No objects found at s3://{bucket}/{s3_path}")
    except Exception as e:
        print(f"  ⚠️  {s3_path}: {e}")
"""

# ============================================================================
# 9. VERIFICATION
# ============================================================================
print("\n" + "=" * 80)
print("CLEANUP VERIFICATION")
print("=" * 80)

# Check endpoints
endpoints = sm_client.list_endpoints()["Endpoints"]
active_endpoints = [e["EndpointName"] for e in endpoints if e["EndpointStatus"] != "Deleting"]
print(f"\n🎯 Active Endpoints: {active_endpoints or '✅ NONE'}")

# Check monitoring schedules
schedules = sm_client.list_monitoring_schedules()["MonitoringScheduleSummaries"]
active_schedules = [s["MonitoringScheduleName"] for s in schedules]
print(f"📋 Monitoring Schedules: {active_schedules or '✅ NONE'}")

# Check alarms
alarms = cw_client.describe_alarms(AlarmNamePrefix="XGB-")["MetricAlarms"]
alarm_names = [a["AlarmName"] for a in alarms]
print(f"⏰ XGB Alarms: {alarm_names or '✅ NONE'}")

# Check models
models = sm_client.list_models()['Models']
xgb_models = [m['ModelName'] for m in models if 'xgb' in m['ModelName'].lower()]
print(f"🤖 XGBoost Models: {xgb_models or '✅ NONE'}")

# ============================================================================
# 10. COST SUMMARY
# ============================================================================
print("\n" + "=" * 80)
print("💰 COST IMPACT")
print("=" * 80)
print("\n✅ Resources Deleted (No Longer Billing):")
print("  • SageMaker Endpoint (ml.m5.large): ~$0.115/hour → $0")
print("  • Monitoring Jobs (when they run): ~$0.024/hour → $0")
print("  • CloudWatch Alarms: ~$0.10/alarm/month → $0")

print("\n⚠️  Remaining Costs (Minimal):")
print("  • S3 Storage: ~$0.023/GB/month (if data kept)")
print("  • CloudWatch Metrics: ~$0.30/metric/month (Standard tier)")
print("  • CloudWatch Logs: ~$0.50/GB ingested")

print("\n💡 To Minimize All Costs:")
print("  1. ✅ Endpoints deleted (DONE)")
print("  2. ✅ Monitoring schedules deleted (DONE)")
print("  3. ⏸️  S3 data kept (uncomment S3 cleanup code to delete)")
print("  4. ⏸️  Model artifacts kept (no ongoing cost)")

print("\n" + "=" * 80)
print("✅ CLEANUP COMPLETE")
print("=" * 80)

AWS RESOURCE CLEANUP - Minimizing Costs

📋 Step 1: Stopping Monitoring Schedules...
  ⏸️  Stopped: xgb-data-quality-schedule
  ⏸️  Stopped: xgb-model-quality-schedule
   Request 240: prediction=0.2150115668773651,0, actual=0.0

🗑️  Step 2: Deleting Monitoring Schedules...
  ⚠️  xgb-data-quality-schedule: An error occurred (ValidationException) when calling the DeleteMonitoringSchedule operation: Monitoring schedule in status Pending
  ⚠️  xgb-model-quality-schedule: An error occurred (ValidationException) when calling the DeleteMonitoringSchedule operation: Monitoring schedule in status Pending

🎯 Step 3: Deleting Endpoint (Stops Instance Billing)...
  ⚠️  Endpoint: An error occurred (ValidationException) when calling the DeleteEndpoint operation: The Endpoint currently has one or more MonitoringSchedules. Please delete the MonitoringSchedules before deleting the Endpoint.

⚙️  Step 4: Deleting Endpoint Configuration...
  ✅ Deleted endpoint config: xgb-benchmark-endpoint-20260215-04500